In [3]:
import numpy as np
import pandas as pd

cars = pd.read_csv("prepared_car_sales.csv")
np.random.seed(1)

from sklearn.model_selection import train_test_split
train_set_df, test_set_df = train_test_split(cars, test_size=0.15 , random_state=0)
labels = train_set_df['Price']
test_labels = test_set_df['Price']


In [2]:
from sklearn.ensemble import RandomForestRegressor, IsolationForest, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from lightgbm import LGBMRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import TransformedTargetRegressor, ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_percentage_error as mape, root_mean_squared_error as rmse

class ExclusiveCars(BaseEstimator, TransformerMixin):
    def __init__(self, brand_col, ex_cars):
        self.brand_col = brand_col
        self.ex_cars = ex_cars
    def fit(self, X, y=None):
        self._ex_cars_set_ = set(self.ex_cars)
        return self
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_copy = X.copy()
            if isinstance(self.brand_col, int):
                col_name = X_copy.columns[self.brand_col]
            else:
                col_name = self.brand_col

            X_copy[col_name] = X_copy[col_name].isin(self._ex_cars_set_).astype(int)
            return X_copy

        X_arr = np.asarray(X).copy()
        X_arr[:, self.brand_col] = np.isin(
            X_arr[:, self.brand_col],
            list(self._ex_cars_set_)
        ).astype(int)
        return X_arr

class CurrencyConverter(BaseEstimator, TransformerMixin):
    def __init__(self, rules: dict = None):
        self.rules = rules
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        if 'Price' not in X_copy.columns or 'Currency' not in X_copy.columns:
            return X_copy
    
        X_copy['Price'] = X_copy['Price'].astype(float)
        rates = X_copy['Currency'].map(self.rules)
        mask = rates.notna()
        X_copy.loc[mask, 'Price'] = X_copy.loc[mask, 'Price'] * rates[mask].astype(float)
        
        return X_copy

class YearsExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, current_year: int = None, target_col: str = 'Production_year', out_col: str = 'Years'):
        self.current_year = current_year if current_year is not None else datetime.now().year
        self.target_col = target_col
        self.out_col = out_col
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        if self.target_col not in X.columns:
            return X
        prod = pd.to_numeric(X[self.target_col], errors='coerce')
        X[self.out_col] = (self.current_year - prod).where(prod.notna(), np.nan).astype(float)
        return X

class OutlierFlagTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_estimators=100, max_samples='auto', contamination=0.05, random_state=42):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.contamination = contamination
        self.random_state = random_state
        self.iso_ = None
    def fit(self, X, y=None):
        self.iso_ = IsolationForest(
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            contamination=self.contamination,
            random_state=self.random_state
        )
        self.iso_.fit(X)
        return self
    def transform(self, X):
        flags = self.iso_.predict(X)
        return flags.reshape(-1, 1)

In [67]:
num_attributes = ['Years', 'Mileage_km', 'Power_HP', 'CO2_emissions', 'Vehicle_brand']
cat_attributes = ['Condition', 'Transmission', 'Type', 'Drive', 'Fuel_type']
ex_cars = [
    "Porsche",
    "Ferrari",
    "McLaren",
    "Bentley",
    "Lamborghini",
    "BMW",
    "Mercedes-Benz",
    "Land Rover",
    "Rolls-Royce"
]
rules = {'EUR': 4.26}

num_pipeline = Pipeline([
    ('exclusive-cars', ExclusiveCars(num_attributes.index('Vehicle_brand'), ex_cars)),
    ('imputer', SimpleImputer(strategy='median')),
    ('standard-scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one-hot-encoder', OneHotEncoder()),
])

coltrans = ColumnTransformer(transformers=[
    ('nums', num_pipeline, num_attributes),
    ('cats', cat_pipeline, cat_attributes),
], 
remainder='drop',
n_jobs=1)

preproc_pipeline = Pipeline([
    ('currency', CurrencyConverter(rules)),
    ('years', YearsExtractor(current_year=2022)),
    ('coltrans', coltrans),
], verbose=True)


#########################################

best_params = {'n_estimators': 1543, 'max_depth': 19, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 10, 'bootstrap': True}
base = RandomForestRegressor(**best_params)
rf_model = TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)

best_params = {'subsample': 0.6, 'reg_lambda': 0.0, 'reg_alpha': 0.1, 'num_leaves': 15, 'n_estimators': 1200,
               'min_child_samples': 30, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
lgbm_model = LGBMRegressor(**best_params)

best_params = {'C': 2808, 'kernel': 'rbf', 'gamma': 'scale', 'epsilon': 3}
svr_model = SVR()

from sklearn.preprocessing import RobustScaler
svr_x_pipeline = Pipeline([
    #('scaler', StandardScaler()),
    #('scaler', RobustScaler()), 
    ('svr', svr_model)
])

from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer
svr_wrapped = TransformedTargetRegressor(
    regressor=svr_x_pipeline,
    transformer=QuantileTransformer(output_distribution='normal', n_quantiles=500)
    #transformer=FunctionTransformer(func=np.log1p, inverse_func=np.expm1)
    #transformer = PowerTransformer(method='yeo-johnson')
)

In [29]:
svr_pipeline = Pipeline([
     ('preproc', preproc_pipeline), 
    # ('add_outlier_feature', FeatureUnion([
    #     ('original_data', FunctionTransformer(lambda x: x)),
    #     ('outlier_flag', OutlierFlagTransformer(contamination=0.10, random_state=42))
    # ])),
    #('final_scaler', StandardScaler()),
    ('svr', svr_wrapped)
])


# from sklearn.model_selection import cross_val_predict, cross_val_score
# # dane = svr_pipeline.fit_transform(train_set_df)
# # for num in dane[0]: print(num)
# test_pipe = Pipeline([
#     ('p', preproc_pipeline),
#     ('s', StandardScaler()),
#     ('svr', svr_model)
# ])

In [68]:
iso = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.15,
    random_state=42
)

cls = Pipeline([
    ('pre', preproc_pipeline),
    ('iso', iso)
])

cls.fit(train_set_df)
is_outliner = cls.predict(train_set_df)
train_set_clean = train_set_df[is_outliner == 1]
labels_clean = labels[is_outliner == 1]
#train_set_clean = preproc_pipeline.fit_transform(train_set_clean)

[Pipeline] .......... (step 1 of 3) Processing currency, total=   0.0s
[Pipeline] ............. (step 2 of 3) Processing years, total=   0.0s
[Pipeline] .......... (step 3 of 3) Processing coltrans, total=   0.0s


In [69]:
#print(len(train_set_clean))
X = preproc_pipeline.fit_transform(train_set_clean)

[Pipeline] .......... (step 1 of 3) Processing currency, total=   0.0s
[Pipeline] ............. (step 2 of 3) Processing years, total=   0.0s
[Pipeline] .......... (step 3 of 3) Processing coltrans, total=   0.0s


In [70]:
import optuna
from sklearn.model_selection import cross_val_predict
def objective(trial):
    C = trial.suggest_int('C', 1000, 3000)
    #kernel = trial.suggest_categorical('kernel', ['rbf'])
    gamma = trial.suggest_float('gamma', 1e-3, 0.01)
    #gamma = trial.suggest_categorical('gamma', ['scale', 'auto', 0.005, 0.01, 0.05, 0.1, 0.2, 0.5])
    epsilon = trial.suggest_float('epsilon', 1, 5)
    #iso_contamination = trial.suggest_float('contamination', 0.01, 0.2)
    n_quantiles = trial.suggest_int('n_quantiles', 250, 1000)

    svr_wrapped.set_params(
        #add_outlier_feature__outlier_flag__contamination=iso_contamination,
        regressor__svr__C=C,
        regressor__svr__epsilon=epsilon,
        regressor__svr__gamma=gamma,
        transformer__n_quantiles=n_quantiles
        #svr__regressor__svr__kernel=kernel

        # C=C,
        # epsilon=epsilon,
        # gamma=gamma,
        # kernel=kernel

        # svr__C=C,
        # svr__epsilon=epsilon,
        # svr__gamma=gamma,
        # svr__kernel=kernel
    )

    # score = cross_val_score(model, train_set, labels, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    # rmse_score = float(np.mean(np.sqrt(-score)))
    #cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    preds = cross_val_predict(svr_wrapped, X, labels_clean, cv=5, n_jobs=-1)
    rmse_score = rmse(labels_clean, preds)
    mape_score = mape(labels_clean, preds)
    trial.set_user_attr("rmse", float(rmse_score))
    trial.set_user_attr("mape", float(1 - mape_score))
    #return float(rmse_score)
    return float(1 - mape_score)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler(seed=42))
study.optimize(objective, n_trials=10)
best_params = study.best_params
print(f"Best Hyperparameters: {best_params}")
best_score = study.best_value
print(f"Best Accuracy: {best_score:.3f}")

[I 2026-03-08 22:24:10,569] A new study created in memory with name: no-name-077a032a-5b40-4a64-871b-17c35548c08a
[I 2026-03-08 22:24:10,711] Trial 0 finished with value: 0.41061409281756245 and parameters: {'C': 1749, 'gamma': 0.009556428757689247, 'epsilon': 3.9279757672456204, 'n_quantiles': 699}. Best is trial 0 with value: 0.41061409281756245.
[I 2026-03-08 22:24:11,105] Trial 1 finished with value: 0.6179635336209455 and parameters: {'C': 1312, 'gamma': 0.002403950683025824, 'epsilon': 1.2323344486727978, 'n_quantiles': 900}. Best is trial 1 with value: 0.6179635336209455.
[I 2026-03-08 22:24:11,972] Trial 2 finished with value: 0.648429469467181 and parameters: {'C': 2202, 'gamma': 0.00737265320016441, 'epsilon': 1.0823379771832098, 'n_quantiles': 978}. Best is trial 2 with value: 0.648429469467181.
[I 2026-03-08 22:24:12,317] Trial 3 finished with value: 0.5365109544176205 and parameters: {'C': 2665, 'gamma': 0.002911051996104486, 'epsilon': 1.7272998688284025, 'n_quantiles': 3

Best Hyperparameters: {'C': 2202, 'gamma': 0.00737265320016441, 'epsilon': 1.0823379771832098, 'n_quantiles': 978}
Best Accuracy: 0.648


In [64]:
#X = preproc_pipeline.fit_transform(train_set_df)
X = preproc_pipeline.fit_transform(train_set_clean)

[Pipeline] .......... (step 1 of 3) Processing currency, total=   0.0s
[Pipeline] ............. (step 2 of 3) Processing years, total=   0.0s
[Pipeline] .......... (step 3 of 3) Processing coltrans, total=   0.0s


In [65]:
def objective(trial):
    C = trial.suggest_int('C', 2000, 3000)
    #kernel = trial.suggest_categorical('kernel', ['rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto', 0.005, 0.01, 0.05, 0.1, 0.2, 0.5])
    epsilon = trial.suggest_int('epsilon', 1, 5)
    
    model = SVR(
        C=C,
        #kernel = kernel,
        gamma = gamma,
        epsilon = epsilon,
    )

    # score = cross_val_score(model, train_set, labels, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    # rmse_score = float(np.mean(np.sqrt(-score)))
    #cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    preds = cross_val_predict(model, X, labels_clean, cv=5, n_jobs=-1)
    rmse_score = rmse(labels_clean, preds)
    mape_score = mape(labels_clean, preds)
    trial.set_user_attr("rmse", float(rmse_score))
    trial.set_user_attr("mape", float(1 - mape_score))
    #return float(rmse_score)
    return float(1 - mape_score)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler(seed=42))
study.optimize(objective, n_trials=10)
best_params = study.best_params
print(f"Best Hyperparameters: {best_params}")
best_score = study.best_value
print(f"Best Accuracy: {best_score:.3f}")

[I 2026-03-08 22:22:07,459] A new study created in memory with name: no-name-f1b25b21-31f1-4b01-891d-6c281b1239ae
[I 2026-03-08 22:22:15,625] Trial 0 finished with value: 0.7318189176468031 and parameters: {'C': 2374, 'gamma': 'scale', 'epsilon': 4}. Best is trial 0 with value: 0.7318189176468031.
[I 2026-03-08 22:22:23,372] Trial 1 finished with value: 0.7286522670233615 and parameters: {'C': 2020, 'gamma': 'scale', 'epsilon': 2}. Best is trial 0 with value: 0.7318189176468031.
[I 2026-03-08 22:22:29,534] Trial 2 finished with value: 0.7247734170036373 and parameters: {'C': 2612, 'gamma': 0.05, 'epsilon': 1}. Best is trial 0 with value: 0.7318189176468031.
[I 2026-03-08 22:22:36,260] Trial 3 finished with value: 0.6852267116140691 and parameters: {'C': 2608, 'gamma': 0.01, 'epsilon': 3}. Best is trial 0 with value: 0.7318189176468031.
[I 2026-03-08 22:22:42,521] Trial 4 finished with value: 0.6022133922670929 and parameters: {'C': 2122, 'gamma': 0.005, 'epsilon': 1}. Best is trial 0 w

Best Hyperparameters: {'C': 2374, 'gamma': 'scale', 'epsilon': 4}
Best Accuracy: 0.732


In [53]:
X = preproc_pipeline.fit_transform(train_set_df)

[Pipeline] .......... (step 1 of 3) Processing currency, total=   0.0s
[Pipeline] ............. (step 2 of 3) Processing years, total=   0.0s
[Pipeline] .......... (step 3 of 3) Processing coltrans, total=   0.0s


In [54]:
from sklearn.model_selection import cross_validate

def display_scores(scores):
    print('Model Performance')
    print('scores: ', scores)
    print('mean: ', scores.mean())
    print('standard deviation: ', scores.std())

best_params = {'C': 2374, 'gamma': 'scale', 'epsilon': 4}
svmreg = SVR(**best_params)
svmreg.fit(X, labels)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error"
}
res = cross_validate(svmreg, X, labels, cv=10, scoring=scoring, return_train_score=True, n_jobs=-1)
mape_scores_hyper = -res["test_mape"]
rmse_scores_hyper = -res["test_rmse"]

display_scores(mape_scores_hyper)
print(1 - mape_scores_hyper.mean())
display_scores(rmse_scores_hyper)

Model Performance
scores:  [0.27301753 0.29272039 0.28950421 0.2826647  0.28367952 0.33812067
 0.2757808  0.31916233 0.27943631 0.28840645]
mean:  0.2922492903916161
standard deviation:  0.0195488350272723
0.7077507096083839
Model Performance
scores:  [52574.62461879 81657.80894451 50134.38685544 60257.5426109
 53723.34823108 63361.91023647 55803.49077155 35560.97642821
 39741.42047573 44306.79459268]
mean:  53712.23037653504
standard deviation:  12443.854593238579
